In [1]:
import torch
import sys

print("Python Version:", sys.version)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Change runtime to T4 GPU.")

Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU: Tesla T4


In [2]:
!pip install -q onnx onnxscript onnxruntime

In [3]:
!pip install -q tensorrt-cu12

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import onnx
import onnxruntime as ort

print("ONNX Version:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)

try:
    import tensorrt as trt
    print("TensorRT Version:", trt.__version__)
except Exception as e:
    print("TensorRT installation issue:", e)

ONNX Version: 1.23.2
ONNX Runtime: 1.32.0
TensorRT Version: 11.5.0.63


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import os

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [6]:
class CNNModel(nn.Module):

    def __init__(self):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)

        self.fc1 = nn.Linear(64 * 5 * 5, 128)

        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.relu(self.conv1(x))
        x = self.pool(x)

        x = self.relu(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))

        x = self.fc2(x)

        return x


model = CNNModel().to(device)

print(model)

CNNModel(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=1600, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [7]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.69MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.24MB/s]

Training samples: 60000
Testing samples: 10000


In [8]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 3

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

print("Training completed!")

Epoch [1/3], Loss: 0.1400
Epoch [2/3], Loss: 0.0443
Epoch [3/3], Loss: 0.0301
Training completed!


In [9]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.78%


In [10]:
os.makedirs("artifacts", exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epochs": epochs,
    "accuracy": accuracy,
    "architecture": "CNNModel"
}

torch.save(
    checkpoint,
    "artifacts/model_checkpoint.pth"
)

print("Checkpoint saved successfully!")

Checkpoint saved successfully!


In [11]:
loaded_checkpoint = torch.load(
    "artifacts/model_checkpoint.pth",
    map_location=device,
    weights_only=True
)

restored_model = CNNModel().to(device)

restored_model.load_state_dict(
    loaded_checkpoint["model_state_dict"]
)

restored_model.eval()

print("Checkpoint loaded successfully!")
print("Stored Accuracy:", loaded_checkpoint["accuracy"])

Checkpoint loaded successfully!
Stored Accuracy: 98.78


In [14]:
torch.onnx.export(
    restored_model,
    dummy_input,
    "artifacts/model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    dynamo=False
)

print("ONNX model saved!")

ONNX model saved!


/tmp/ipykernel_531/3670343116.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [18]:
onnx_model = onnx.load("artifacts/model.onnx")

onnx.checker.check_model(onnx_model)

print("ONNX graph validation successful!")

print("Graph Inputs:")
for item in onnx_model.graph.input:
    print(item.name)

print("Graph Outputs:")
for item in onnx_model.graph.output:
    print(item.name)

ONNX graph validation successful!
Graph Inputs:
input
Graph Outputs:
output


In [20]:
!pip install -q netron

In [24]:
import netron

netron.start("artifacts/model.onnx")

('localhost', 8081)

In [25]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(
    "artifacts/model.onnx",
    providers=["CPUExecutionProvider"]
)

sample_input = torch.randn(1, 1, 28, 28)

pytorch_output = restored_model(
    sample_input.to(device)
).detach().cpu().numpy()

onnx_output = session.run(
    ["output"],
    {"input": sample_input.numpy()}
)[0]

print("PyTorch Prediction:", np.argmax(pytorch_output))
print("ONNX Prediction:", np.argmax(onnx_output))

print(
    "Outputs match:",
    np.allclose(
        pytorch_output,
        onnx_output,
        atol=1e-4
    )
)

PyTorch Prediction: 2
ONNX Prediction: 2
Outputs match: True


In [26]:
import tensorrt as trt

print("TensorRT Version:", trt.__version__)

assert torch.cuda.is_available(), "Enable a GPU runtime first!"

logger = trt.Logger(trt.Logger.WARNING)

builder = trt.Builder(logger)

print("TensorRT Builder initialized successfully!")

TensorRT Version: 11.5.0.63
TensorRT Builder initialized successfully!


In [29]:
import tensorrt as trt

ONNX_PATH = "artifacts/model.onnx"
ENGINE_PATH = "artifacts/model.engine"

logger = trt.Logger(trt.Logger.WARNING)

builder = trt.Builder(logger)

# Modern TensorRT: explicit batch is the default
network = builder.create_network(0)

parser = trt.OnnxParser(network, logger)

# Read ONNX file
with open(ONNX_PATH, "rb") as model_file:

    model_data = model_file.read()

    if not parser.parse(model_data):

        print("ONNX parsing failed!")

        for i in range(parser.num_errors):
            print(parser.get_error(i))

        raise RuntimeError("ONNX parsing failed!")

print("ONNX graph parsed successfully!")
print("Number of layers:", network.num_layers)

ONNX graph parsed successfully!
Number of layers: 18


In [31]:
import tensorrt as trt

config = builder.create_builder_config()

config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE,
    1 << 30
)

# Enable FP16 only if supported by the installed TensorRT API
if hasattr(trt.BuilderFlag, "FP16"):

    config.set_flag(trt.BuilderFlag.FP16)

    print("FP16 mode enabled")

else:

    print("FP16 unavailable. Using FP32.")

print("TensorRT configuration completed!")

FP16 unavailable. Using FP32.
TensorRT configuration completed!


In [32]:
serialized_engine = builder.build_serialized_network(
    network,
    config
)

if serialized_engine is None:

    raise RuntimeError(
        "TensorRT engine generation failed!"
    )

with open(ENGINE_PATH, "wb") as f:

    f.write(serialized_engine)

print("TensorRT engine generated successfully!")
print("Saved at:", ENGINE_PATH)

TensorRT engine generated successfully!
Saved at: artifacts/model.engine


In [33]:
import os

if os.path.exists(ENGINE_PATH):

    print("TensorRT Engine Found!")

    print(
        "Engine Size:",
        round(os.path.getsize(ENGINE_PATH) / 1024, 2),
        "KB"
    )

else:

    print("Engine generation unsuccessful!")

TensorRT Engine Found!
Engine Size: 949.23 KB


In [34]:
runtime = trt.Runtime(logger)

with open(ENGINE_PATH, "rb") as f:

    engine_data = f.read()

engine = runtime.deserialize_cuda_engine(
    engine_data
)

if engine is None:

    raise RuntimeError("Engine deserialization failed!")

print("TensorRT Engine Loaded Successfully!")
print("Number of I/O tensors:", engine.num_io_tensors)

TensorRT Engine Loaded Successfully!
Number of I/O tensors: 2


In [35]:
import os

artifact_folder = "artifacts"

print("MODEL ARTEFACT LIFECYCLE")
print("-" * 40)

for filename in os.listdir(artifact_folder):

    filepath = os.path.join(
        artifact_folder,
        filename
    )

    size = os.path.getsize(filepath) / 1024

    print(
        f"{filename:<30} {size:.2f} KB"
    )

MODEL ARTEFACT LIFECYCLE
----------------------------------------
model_checkpoint.pth           2647.40 KB
model.onnx                     880.63 KB
model.engine                   949.23 KB


In [37]:
!zip -r model_lifecycle_artifacts.zip artifacts/

files.download("model_lifecycle_artifacts.zip")

  adding: artifacts/ (stored 0%)
  adding: artifacts/model_checkpoint.pth (deflated 17%)
  adding: artifacts/model.onnx (deflated 7%)
  adding: artifacts/model.engine (deflated 10%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>